In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/speech_project', exist_ok=True)

Mounted at /content/drive


In [ ]:
!pip install -q librosa tensorflow-datasets plotly streamlit

In [ ]:
import os, json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import librosa
import tensorflow_datasets as tfds
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
SAMPLE_RATE   = 16000
N_MFCC        = 40
HOP_LENGTH    = 512
NUM_CLASSES   = 10          # ← keep small so training finishes in ~20 min on CPU
BATCH_SIZE    = 32
EPOCHS        = 15
MODEL_PATH  = "/content/drive/MyDrive/speech_project/model.keras"
LABELS_PATH = "/content/drive/MyDrive/speech_project/labels.json"

# 10 easy, distinct words to classify
TARGET_WORDS  = ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"]

# ── Audio helpers ─────────────────────────────────────────────────────────────
def pad_or_trim(audio, length=SAMPLE_RATE):
    if len(audio) < length:
        audio = np.pad(audio, (0, length - len(audio)))
    return audio[:length]

def extract_mfcc(audio):
    mfcc = librosa.feature.mfcc(
        y=audio.astype(np.float32),
        sr=SAMPLE_RATE,
        n_mfcc=N_MFCC,
        hop_length=HOP_LENGTH
    )
    return mfcc[..., np.newaxis]       # (40, T, 1)

# ── Dataset ───────────────────────────────────────────────────────────────────
def load_data():
    print("Downloading dataset (first run only, ~2 GB)…")
    ds = tfds.load("speech_commands", split="train", as_supervised=False)

    X, y = [], []
    label2idx = {w: i for i, w in enumerate(TARGET_WORDS)}

    for ex in tqdm(ds, desc="Loading"):
        label = ex["label"].numpy()
        # tfds gives integer label; get string via builder info
        pass  # handled below

    # Reload with full info to get label names
    builder = tfds.builder("speech_commands")
    builder.download_and_prepare()
    info = builder.info
    label_names = info.features["label"].names   # list of all class strings

    ds = builder.as_dataset(split="train", as_supervised=False)
    for ex in tqdm(ds, desc="Filtering"):
        word = label_names[ex["label"].numpy()]
        if word not in label2idx:
            continue
        audio = ex["audio"].numpy().astype(np.float32) / 32768.0  # int16 → float
        audio = pad_or_trim(audio)
        X.append(extract_mfcc(audio))
        y.append(label2idx[word])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(input_shape, num_classes):
    model = keras.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),     # no Flatten needed → fewer params
        layers.Dropout(0.4),

        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ── Train ─────────────────────────────────────────────────────────────────────
def main():
    X, y = load_data()
    print(f"\nLoaded {len(X)} samples  |  shape: {X.shape}")

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    model = build_model(X.shape[1:], NUM_CLASSES)
    model.summary()

    callbacks = [
        keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2, verbose=1),
    ]

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        callbacks=callbacks,
    )

    model.save(MODEL_PATH)
    with open(LABELS_PATH, "w") as f:
        json.dump(TARGET_WORDS, f)

    loss, acc = model.evaluate(X_val, y_val, verbose=0)
    print(f"\nSaved → {MODEL_PATH}")
    print(f"   Val accuracy: {acc:.1%}")

if __name__ == "__main__":
    main()

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 40, 32, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 40, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 20, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 20, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 20, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 20, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 8, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 8, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 10, 8, 128)     │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 10, 8, 128)     │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 111,370 (435.04 KB)

 Trainable params: 110,922 (433.29 KB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/15
770/770 ━━━━━━━━━━━━━━━━━━━━ 115s 143ms/step - accuracy: 0.5909 - loss: 1.1641 - val_accuracy: 0.8146 - val_loss: 0.5389 - learning_rate: 0.0010
Epoch 2/15
770/770 ━━━━━━━━━━━━━━━━━━━━ 101s 131ms/step - accuracy: 0.8034 - loss: 0.5736 - val_accuracy: 0.8848 - val_loss: 0.3222 - learning_rate: 0.0010
Epoch 3/15
770/770 ━━━━━━━━━━━━━━━━━━━━ 141s 130ms/step - accuracy: 0.8464 - loss: 0.4561 - val_accuracy: 0.9097 - val_loss: 0.2561 - learning_rate: 0.0010
Epoch 4/15
770/770 ━━━━━━━━━━━━━━━━━━━━ 101s 131ms/step - accuracy: 0.8629 - loss: 0.4085 - val_accuracy: 0.9144 - val_loss: 0.2371 - learning_rate: 0.0010
Epoch 5/15
770/770 ━━━━━━━━━━━━━━━━━━━━ 103s 133ms/step - accuracy: 0.8756 - loss: 0.3722 - val_accuracy: 0.9227 - val_loss: 0.2224 - learning_rate: 0.0010
Epoch 6/15
770/770 ━━━━━━━━━━━━━━━━━━━━ 109s 141ms/step - accuracy: 0.8835 - loss: 0.3484 - val_accuracy: 0.9279 - val_loss: 0.2002 - learning_rate: 0.0010
Epoch 7/15
770/770 ━━━━━━━━━━━━━━━━━━━━ 101s 131ms/step - accura

In [ ]:
import json, numpy as np, librosa, tensorflow as tf

model  = tf.keras.models.load_model('/content/drive/MyDrive/speech_project/model.keras')
labels = json.load(open('/content/drive/MyDrive/speech_project/labels.json'))

print("✅ Model loaded")
print("Classes:", labels)
model.summary()

✅ Model loaded
Classes: ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go']


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 40, 32, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 40, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 20, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 20, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 20, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 20, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 8, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 8, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 10, 8, 128)     │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 10, 8, 128)     │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 333,216 (1.27 MB)

 Trainable params: 110,922 (433.29 KB)

 Non-trainable params: 448 (1.75 KB)

 Optimizer params: 221,846 (866.59 KB)